In [58]:
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from typing import Final
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [59]:
window_size: Final[int] = 1

EPOCHS: Final[int] = 50
BATCH_SIZE: Final[int] = 32

In [61]:
df = pd.read_csv("DailyDelhiClimateTrain.csv")

df['date'] = pd.to_datetime(df['date'])
print(f"Shape: {df.shape}")

display(df)

Shape: (1462, 5)


,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.000000,84.500000,0.000000,1015.666667
1,2013-01-02,7.400000,92.000000,2.980000,1017.800000
2,2013-01-03,7.166667,87.000000,4.633333,1018.666667
3,2013-01-04,8.666667,71.333333,1.233333,1017.166667
4,2013-01-05,6.000000,86.833333,3.700000,1016.500000
...,...,...,...,...,...
1457,2016-12-28,17.217391,68.043478,3.547826,1015.565217
1458,2016-12-29,15.238095,87.857143,6.000000,1016.904762
1459,2016-12-30,14.095238,89.666667,6.266667,1017.904762
1460,2016-12-31,15.052632,87.000000,7.325000,1016.100000


In [62]:
df.info()
print(f"\nduplicates = {df.duplicated().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1462 non-null   datetime64[ns]
 1   meantemp      1462 non-null   float64       
 2   humidity      1462 non-null   float64       
 3   wind_speed    1462 non-null   float64       
 4   meanpressure  1462 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 57.2 KB

duplicates = 0


In [63]:
fraction_missing: int = 0.1

mask = np.random.rand(*df.shape) < fraction_missing
df.loc[:, list(df.columns)][mask] = np.nan

In [64]:
df.interpolate(method='linear', inplace=True)

In [65]:
numeric_cols: list = df.select_dtypes(include=[np.number]).columns

noise = np.random.normal(0, 0.02, df[numeric_cols].shape)
df[numeric_cols] = df[numeric_cols] + noise

In [72]:
series = df["humidity"].values.reshape(-1, 1)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(series)

In [73]:
def get_dataset(data, window_size):
    x, y = [], []
    for i in range(len(data) - window_size - 1):
        x.append(data[i:i + window_size])
        y.append(data[i + window_size + 1])
    return np.array(x), np.array(y)

x, y = get_dataset(data_scaled, window_size)
x = x[..., None]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

In [74]:
rnn_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.SimpleRNN(64),
    layers.Dense(1)
])
rnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = rnn_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0250 - mae: 0.1236 - val_loss: 0.0150 - val_mae: 0.0931
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0144 - mae: 0.0936 - val_loss: 0.0150 - val_mae: 0.0933
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0931 - val_loss: 0.0156 - val_mae: 0.0941
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0927 - val_loss: 0.0158 - val_mae: 0.0944
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0925 - val_loss: 0.0155 - val_mae: 0.0936
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0924 - val_loss: 0.0156 - val_mae: 0.0938
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0927 - val_loss: 0.0164 - val_mae: 0.0960
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 0.0920 - val_loss: 0.0153 - val_mae: 0.0933
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0142 - mae: 

In [81]:
def evaluate_model(model, x_test, y_test):
    predictions = model.predict(x_test)
    return {
        "MSE": mean_squared_error(y_test, predictions),
        "MAE": mean_absolute_error(y_test, predictions),
        "R2": r2_score(y_test, predictions),
    }

In [ ]:
print(evaluate_model(rnn_model, x_test, y_test))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
{'MSE': 0.015241606652267713, 'MAE': 0.09329112528882623, 'R2': 0.5348193921785737}


In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.LSTM(64),
    layers.Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = lstm_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.2307 - mae: 0.4391 - val_loss: 0.1555 - val_mae: 0.3615
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0838 - mae: 0.2459 - val_loss: 0.0420 - val_mae: 0.1733
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0253 - mae: 0.1277 - val_loss: 0.0183 - val_mae: 0.1059
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0196 - mae: 0.1116 - val_loss: 0.0177 - val_mae: 0.1040
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0191 - mae: 0.1102 - val_loss: 0.0174 - val_mae: 0.1030
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0186 - mae: 0.1087 - val_loss: 0.0170 - val_mae: 0.1018
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0182 - mae: 0.1075 - val_loss: 0.0168 - val_mae: 0.1010
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0177 - mae: 0.1059 - val_loss: 0.0164 - val_mae: 0.0996
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0172 - mae:

In [ ]:
print(evaluate_model(lstm_model, x_test, y_test))

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
MSE: 0.015456866152069829
MAE: 0.09390456726093326
R2: 0.5282495765914232


In [ ]:
gru_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.GRU(64),
    layers.Dense(1)
])
gru_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = gru_model.fit(
    x_train, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_data=(x_test, y_test)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.1387 - mae: 0.3177 - val_loss: 0.0317 - val_mae: 0.1472
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0231 - mae: 0.1208 - val_loss: 0.0185 - val_mae: 0.1068
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0205 - mae: 0.1143 - val_loss: 0.0181 - val_mae: 0.1055
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0193 - mae: 0.1108 - val_loss: 0.0176 - val_mae: 0.1037
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0183 - mae: 0.1079 - val_loss: 0.0169 - val_mae: 0.1014
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0174 - mae: 0.1050 - val_loss: 0.0161 - val_mae: 0.0986
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0166 - mae: 0.1025 - val_loss: 0.0160 - val_mae: 0.0977
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0159 - mae: 0.1000 - val_loss: 0.0155 - val_mae: 0.0960
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0154 - mae:

In [ ]:
print(evaluate_model(gru_model, x_test, y_test))

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
MSE: 0.015340523779487723
MAE: 0.09340560521881155
R2: 0.5318003975007844


In [86]:
display(pd.DataFrame({'RNN': evaluate_model(rnn_model, x_test, y_test),
                      'LSTM': evaluate_model(lstm_model, x_test, y_test),
                      'GRU': evaluate_model(gru_model, x_test, y_test),
                    }))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,RNN,LSTM,GRU
MSE,0.015242,0.015457,0.015341
MAE,0.093291,0.093905,0.093406
R2,0.534819,0.528250,0.531800
